In [1]:
import sys
import os

# Add project root (one level up from 'notebooks')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
# kurianuthuppu11@gmail.com

token = "VybxF62Uoej4Tv3RvtlSBDeBOAryDKX97yJvsqL2gAB_"

from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token=token,
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/e3a7616824024c57b40a7014c5e27c30:b4f5e1cd-3aba-470f-93e5-f82079969538::",
    channel="ibm_quantum_platform",
    overwrite=True,
    set_as_default=True,
)

In [2]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    EstimatorOptions,
)
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from adaptive_error_mitigation.analytics import (
    extract_basic_features,
    get_qubit_layout_mapping,
    extract_backend_metrics,
    calculate_derived_noise_metrics,
    analyze_qubit_idling,
)
from adaptive_error_mitigation.utils import cal_em_eff_estimator
from adaptive_error_mitigation import adaptive_estimator

In [3]:
# Number of qubits for the GHZ state
NUM_QUBITS = 15

# Create GHZ state
ghz = QuantumCircuit(NUM_QUBITS)
ghz.h(0)
for i in range(NUM_QUBITS - 1):
    ghz.cx(i, i + 1)

# Loschmidt echo circuit: GHZ → barrier → GHZ† → measurement
echo = QuantumCircuit(NUM_QUBITS)
echo.compose(ghz, inplace=True)
echo.barrier()
echo.compose(ghz.inverse(), inplace=True)
echo.x(range(NUM_QUBITS))

# Visualize circuit
# echo.draw("mpl")

In [5]:
# NUM_QUBITS = 2

# distribution = {
#     1: 0.5,
#     2: 0.5,
# }

# qc = random_circuit(
#     num_qubits=NUM_QUBITS,
#     depth=9,
#     max_operands=2,
#     num_operand_distribution=distribution,
#     seed=45,
# )

# # Loschmidt echo circuit: RC → barrier → RC† → measurement
# echo = QuantumCircuit(NUM_QUBITS)
# echo.compose(qc, inplace=True)
# # echo.barrier()
# # echo.compose(qc.inverse(), inplace=True)
# echo.x(range(NUM_QUBITS))

# echo.draw("mpl", fold=-1)

In [4]:
service = QiskitRuntimeService()
backend = service.backend("ibm_torino")
target = backend.target

management.get:WARNING:2025-11-24 20:33:26,401: Loading default saved account


In [5]:
pm_lvl3 = generate_preset_pass_manager(
    optimization_level=3,
    seed_transpiler=42,
    backend=backend,
    scheduling_method="alap",
)
isa_qc = pm_lvl3.run(echo)

In [8]:
# isa_qc.draw("mpl", fold=-1)

In [7]:
pauli_string = "Z" * NUM_QUBITS

# The observable is then created from this string
observable = SparsePauliOp(pauli_string)

isa_observable = observable.apply_layout(isa_qc.layout)

In [9]:
basic_features = extract_basic_features(isa_qc)
print(f"Basic features: {basic_features}")
gate_2q_per_qubit = basic_features["num_2q_gates"] / basic_features["qubits_used"]
print(f"2Q gates per qubit: {gate_2q_per_qubit:.4f}")
gate_2q_density = basic_features["2q_gate_density"]
print(f"2Q gate density: {gate_2q_density:.4f}")

Basic features: {'tot_qubits': 133, 'qubits_used': 15, 'depth': 90, 'size': 332, 'gate_counts': {'delay': 172, 'rz': 74, 'sx': 58, 'cz': 28, 'barrier': 1}, 'norm_gate_dist': {'rz': 0.4625, 'sx': 0.3625, 'cz': 0.175}, 'num_1q_gates': 132, 'num_2q_gates': 28, '2q_gate_density': 0.175, 'num_measurements': 0}
2Q gates per qubit: 1.8667
2Q gate density: 0.1750


In [10]:
qubits_used = extract_basic_features(isa_qc)["qubits_used"]
print(f"Qubits used in the circuit: {qubits_used}")
ons = calculate_derived_noise_metrics(isa_qc, backend)["overall_noise_sensitivity"]
print(f"Overall Noise Sensitivity: {ons}")
t2_avg = extract_backend_metrics(isa_qc, backend)["avg_t2_time"]
print(f"Average T2 Time: {t2_avg}")
duration_dt = isa_qc.duration
dt_in_seconds = backend.dt  # dt is in seconds
duration_ns = duration_dt * dt_in_seconds
print(f"Circuit duration: {duration_ns}")
h_zne = ons + qubits_used * (duration_ns / t2_avg)
print(f"Estimated H_ZNE: {h_zne}")

Qubits used in the circuit: 15
Overall Noise Sensitivity: 0.08600826611494904
Average T2 Time: 0.00017652490273852074
Circuit duration: 2.864e-06
Estimated H_ZNE: 0.3293733626737894


C:\Users\uthupku\AppData\Local\Temp\ipykernel_32520\2289681968.py:7: DeprecationWarning: The property ``qiskit.circuit.quantumcircuit.QuantumCircuit.duration`` is deprecated as of Qiskit 1.3.0. It will be removed in Qiskit 3.0.0.
  duration_dt = isa_qc.duration


In [ ]:
analyze_qubit_idling(isa_qc, backend)["max_ratio_qubit"]["qubit_idx"]

{'qubit_idx': 15,
 't2_dt': 9620.32,
 'idle_dt': 333,
 'decoher_err_prob': 0.034022}

In [11]:
estimator = AerEstimator()
pub = (isa_qc, isa_observable)

job = estimator.run([pub])
result = job.result()
pub_result_ideal = result[0]
evs_ideal = pub_result_ideal.data.evs
print(evs_ideal)

-1.0


In [12]:
job = adaptive_estimator.run([(isa_qc, isa_observable)], backend)

print(f"Job-Id: {job.job_id()}")
result = job.result()

--- Initiating Adaptive Error Mitigation and Suppression Framework ---

--> DEFAULT SETTING: Using Default Precision set to 0.015625 and default shots 4096

---> HEURISTIC TRIGGERED: Readout Error Threshold Exceeded
     | Metric: MAX READOUT ERROR - 0.1367 (on Qubit 0)
     | Threshold Set: 0.0100 (READOUT_ERROR_THRESHOLD (config.py))
---> ACTION TAKEN: ENABLED TREX (Twirled Readout Error eXtinction )
     | **Derived Parameters:** shots_per_randomization set to 128 (Shots: 4096 (DEFAULT_SHOTS (config.py)) / Randomizations: 32 (NUM_RANDOMIZATIONS (config.py)))
---> Circuit already scheduled (duration: 716).

---> HEURISTIC TRIGGERED: Decoherence Error Threshold Exceeded
     | Metric: MAX DECOHERENCE ERROR PROBABILITY - 0.0340 (on Qubit 15)
     | Threshold Set: 0.0010 (DD_ERROR_THRESHOLD (config.py))
---> ACTION TAKEN: ENABLED Dynamic Decoupling (DD)



TranspilerError: 'The input circuit circuit-47 is not scheduled. Call one of scheduling passes before running the PadDynamicalDecoupling pass.'

In [227]:
config_est_options = {
    "dynamical_decoupling":{"enable": False},
    "twirling":{"enable_gates": False, "enable_measure": False},
    "resilience_level":0,
    "resilience":{
        "zne_mitigation": False,
        "zne": {
            "noise_factors": (1, 3, 5),
            "extrapolator": "exponential",
        },
    },
}

estimator_options = EstimatorOptions(**config_est_options)

print(estimator_options)

EstimatorOptions(_VERSION=2, max_execution_time=Unset, environment=EnvironmentOptions(log_level='WARNING', job_tags=None, private=False), simulator=SimulatorOptions(noise_model=Unset, seed_simulator=Unset, coupling_map=Unset, basis_gates=Unset), default_precision=Unset, default_shots=Unset, resilience_level=0, seed_estimator=Unset, dynamical_decoupling=DynamicalDecouplingOptions(enable=False, sequence_type=Unset, extra_slack_distribution=Unset, scheduling_method=Unset, skip_reset_qubits=Unset), resilience=ResilienceOptionsV2(measure_mitigation=Unset, measure_noise_learning=MeasureNoiseLearningOptions(num_randomizations=Unset, shots_per_randomization=Unset), zne_mitigation=False, zne=ZneOptions(amplifier=Unset, noise_factors=(1.0, 3.0, 5.0), extrapolator='exponential', extrapolated_noise_factors=Unset), pec_mitigation=Unset, pec=PecOptions(max_overhead=Unset, noise_gain=Unset), layer_noise_learning=LayerNoiseLearningOptions(max_layers_to_learn=Unset, shots_per_randomization=Unset, num_r

In [228]:
estimator = Estimator(mode=backend, options=estimator_options)

pub = (isa_qc, isa_observable)

job = estimator.run([pub])

print(f"Job-Id: {job.job_id()}")
result = job.result()

Job-Id: d4i3b1h2bisc73a517l0


In [ ]:
# GHZ echo circuit - 3 Qubit with no EM
# job = service.job(job_id="d4i18occdebc73f2im7g")

# GHZ echo circuit - 10 Qubit with no EM
# job = service.job(job_id="d4i1t2h2bisc73a4vlg0")

# GHZ echo circuit - 25 Qubit with no EM
# job = service.job(job_id="d4i20gelo8as739r7dk0")

# GHZ echo circuit - 40 Qubit with no EM
# job = service.job(job_id="d4i2cep2bisc73a505n0")

# GHZ echo circuit - 30 Qubit with no EM
# job = service.job(job_id="d4i2fskcdebc73f2ju1g")

# GHZ echo circuit - 27 Qubit with no EM
# job = service.job(job_id="d4i2ib6lo8as739r82tg")

# GHZ echo circuit - 15 Qubit with no EM
# job = service.job(job_id="d4i2loolslhc73d266lg")

# GHZ echo circuit - 11 Qubit with no EM
# job = service.job(job_id="d4i35ch2bisc73a5115g")

# GHZ echo circuit - 12 Qubit with no EM
# job = service.job(job_id="d4i393kcdebc73f2kqfg")

# GHZ echo circuit - 29 Qubit with no EM
# job = service.job(job_id="d4i3b1h2bisc73a517l0")

In [229]:
result = job.result()
pub_result_NO_ZNE = result[0]
evs_NO_ZNE = pub_result_NO_ZNE.data.evs
print(evs_NO_ZNE)

0.013671875


In [230]:
config_est_options = {
    "dynamical_decoupling": {"enable": False},
    "twirling": {"enable_gates": True, "enable_measure": False},
    "resilience_level": 0,
    "resilience": {
        "zne_mitigation": True,
        "zne": {
            "amplifier": "gate_folding",
            "noise_factors": (1, 3, 5),
            "extrapolator": "exponential",
        },
    },
}

estimator_options = EstimatorOptions(**config_est_options)

print(estimator_options)

EstimatorOptions(_VERSION=2, max_execution_time=Unset, environment=EnvironmentOptions(log_level='WARNING', job_tags=None, private=False), simulator=SimulatorOptions(noise_model=Unset, seed_simulator=Unset, coupling_map=Unset, basis_gates=Unset), default_precision=Unset, default_shots=Unset, resilience_level=0, seed_estimator=Unset, dynamical_decoupling=DynamicalDecouplingOptions(enable=False, sequence_type=Unset, extra_slack_distribution=Unset, scheduling_method=Unset, skip_reset_qubits=Unset), resilience=ResilienceOptionsV2(measure_mitigation=Unset, measure_noise_learning=MeasureNoiseLearningOptions(num_randomizations=Unset, shots_per_randomization=Unset), zne_mitigation=True, zne=ZneOptions(amplifier='gate_folding', noise_factors=(1.0, 3.0, 5.0), extrapolator='exponential', extrapolated_noise_factors=Unset), pec_mitigation=Unset, pec=PecOptions(max_overhead=Unset, noise_gain=Unset), layer_noise_learning=LayerNoiseLearningOptions(max_layers_to_learn=Unset, shots_per_randomization=Unse

In [231]:
estimator = Estimator(mode=backend, options=estimator_options)

pub = (isa_qc, isa_observable)

job = estimator.run([pub])

print(f"Job-Id: {job.job_id()}")
result = job.result()

Job-Id: d4i3baglslhc73d26u7g


In [ ]:
# GHZ echo circuit - 3 Qubit with ZNE
# job = service.job(job_id="d4i1k7h2bisc73a4vcug")

# GHZ echo circuit - 10 Qubit with ZNE
# job = service.job(job_id="d4i1t712bisc73a4vll0")

# GHZ echo circuit - 25 Qubit with ZNE
# job = service.job(job_id="d4i28g8lslhc73d25ng0")

# GHZ echo circuit - 40 Qubit with ZNE
# job = service.job(job_id="d4i2cm0lslhc73d25s80")

# GHZ echo circuit - 30 Qubit with ZNE
# job = service.job(job_id="d4i2g5h2bisc73a509v0")

# GHZ echo circuit - 27 Qubit with ZNE
# job = service.job(job_id="d4i2iiccdebc73f2k15g")

# GHZ echo circuit - 15 Qubit with ZNE
# job = service.job(job_id="d4i2m8glslhc73d2678g")

# GHZ echo circuit - 11 Qubit with ZNE
# job = service.job(job_id="d4i366glslhc73d26ot0")

# GHZ echo circuit - 12 Qubit with ZNE
# job = service.job(job_id="d4i39d92bisc73a515o0")

# GHZ echo circuit - 29 Qubit with ZNE
# job = service.job(job_id="d4i3baglslhc73d26u7g")

In [232]:
result = job.result()
pub_result_with_ZNE = result[0]
evs_withZNE = pub_result_with_ZNE.data.evs
print(evs_withZNE)

0.00020164001369450457


In [234]:
# --- RUN BENCHMARK ---
results = cal_em_eff_estimator(evs_NO_ZNE, evs_withZNE, evs_ideal)

# --- DISPLAY RESULTS ---
print(f"--- Dynamic Decoupling Efficacy Benchmark (Estimator) ---")
print(f"📈 **Absolute Error** (Deviation from 0) (Lower is Better)")
print(f"  No ZNE: {results['ERROR_nodd']:.4f} (EVS: {results['EVS_nodd']:.4f})")
print(f"  With ZNE: {results['ERROR_dd']:.4f} (EVS: {results['EVS_dd']:.4f})")
dev_eff_msg = "🎉 Reduced" if results["ERROR_reduction_percent"] > 0 else "⚠️ Increased"
print(
    f"  Efficacy: {dev_eff_msg} Error by: {abs(results['ERROR_reduction_percent']):.2f}%"
)

print("---------------------------------------------------------------")

--- Dynamic Decoupling Efficacy Benchmark (Estimator) ---
📈 **Absolute Error** (Deviation from 0) (Lower is Better)
  No ZNE: 1.0137 (EVS: 0.0137)
  With ZNE: 1.0002 (EVS: 0.0002)
  Efficacy: 🎉 Reduced Error by: 1.33%
---------------------------------------------------------------
